# Day 19 — K-Means Clustering 🟢

A practical unsupervised learning project using the **Mall Customers** dataset. The project groups customers based on annual income and spending score.

## 🎯 Objectives

- Understand unsupervised learning and clustering
- Learn how K-Means works
- Use the Elbow Method to investigate K
- Scale features before clustering
- Analyze clusters and centroids
- Visualize customer segments
- Use PCA for an additional 2D visualization

## 📊 Dataset

This project uses `Mall_Customers.csv`. Place the CSV in the same folder as this notebook.

Clustering features:
- `Annual Income (k$)`
- `Spending Score (1-100)`

## 🛠️ Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

## 📥 Load Dataset

In [ ]:
df = pd.read_csv("Mall_Customers.csv")

print("Dataset Shape:", df.shape)
display(df.head())

## 🔎 Explore Dataset

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDataset Information:")
df.info()

## 📌 Select Features

In [ ]:
X = df[[
    "Annual Income (k$)",
    "Spending Score (1-100)"
]]

print("Features Used:")
print(X.columns.tolist())

display(X.head())

## 📈 Visualize Original Data

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    X["Annual Income (k$)"],
    X["Spending Score (1-100)"],
    alpha=0.7
)

plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.title("Customer Data Before Clustering")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 📐 Feature Scaling

K-Means is distance-based, so scaling helps put the selected features on comparable scales.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaling completed.")
print(X_scaled[:5])

## 📉 Elbow Method

The Elbow Method evaluates different values of K using **inertia (WCSS)**. The elbow of the curve can help identify a suitable number of clusters.

In [ ]:
inertia = []
K_values = range(1, 11)

for k in K_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    model.fit(X_scaled)
    inertia.append(model.inertia_)

for k, value in zip(K_values, inertia):
    print(f"K = {k} → Inertia = {value:.2f}")

## 📊 Elbow Curve

In [ ]:
plt.figure(figsize=(9, 6))

plt.plot(
    K_values,
    inertia,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia / WCSS")
plt.title("Elbow Method for Selecting K")
plt.xticks(list(K_values))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 🎯 Select K

For this project, **K = 5** is used for the main experiment based on the typical elbow pattern of this dataset.

In [ ]:
optimal_k = 5

kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X_scaled)

print("Number of Clusters:", optimal_k)

## 🏷️ Add Cluster Labels

In [ ]:
df["Cluster"] = cluster_labels

display(df[[
    "CustomerID",
    "Gender",
    "Age",
    "Annual Income (k$)",
    "Spending Score (1-100)",
    "Cluster"
]].head(15))

## 👥 Customers in Each Cluster

In [ ]:
cluster_counts = (
    df["Cluster"]
    .value_counts()
    .sort_index()
)

display(cluster_counts)

## 📍 Cluster Centroids

In [ ]:
centers_original = scaler.inverse_transform(
    kmeans.cluster_centers_
)

centroids = pd.DataFrame(
    centers_original,
    columns=[
        "Annual Income (k$)",
        "Spending Score (1-100)"
    ]
)

centroids.index.name = "Cluster"

display(centroids.round(2))

## 📊 Customer Segmentation Visualization

In [ ]:
plt.figure(figsize=(10, 7))

for cluster in sorted(df["Cluster"].unique()):
    cluster_data = df[df["Cluster"] == cluster]

    plt.scatter(
        cluster_data["Annual Income (k$)"],
        cluster_data["Spending Score (1-100)"],
        label=f"Cluster {cluster}",
        alpha=0.7
    )

plt.scatter(
    centers_original[:, 0],
    centers_original[:, 1],
    marker="X",
    s=200,
    label="Centroids"
)

plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.title("K-Means Customer Segmentation")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 📋 Cluster Summary

In [ ]:
cluster_summary = (
    df.groupby("Cluster")[[
        "Age",
        "Annual Income (k$)",
        "Spending Score (1-100)"
    ]]
    .mean()
    .round(2)
)

display(cluster_summary)

## 🧠 Understanding Customer Segments

Cluster interpretation should be based on the average income and spending score produced above. For example, a high-income/high-spending group may represent a high-value customer segment, while a low-income/low-spending group may represent a lower-spending segment.

## 🗺️ PCA Visualization

PCA is used here only for visualization. The actual K-Means clustering is performed using the two original scaled features.

In [ ]:
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

print("Original Dimensions:", X_scaled.shape[1])
print("PCA Dimensions:", X_pca.shape[1])
print("Variance Explained:", pca.explained_variance_ratio_.sum())

## 📈 K-Means Clusters Using PCA

In [ ]:
plt.figure(figsize=(10, 7))

for cluster in sorted(df["Cluster"].unique()):
    mask = df["Cluster"] == cluster

    plt.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        label=f"Cluster {cluster}",
        alpha=0.7
    )

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("K-Means Clusters Visualized Using PCA")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 💾 Save Clustered Dataset

In [ ]:
output_file = "Mall_Customers_Clustered.csv"

df.to_csv(
    output_file,
    index=False
)

print(f"Clustered dataset saved as: {output_file}")

## 📊 Final Results

In [ ]:
print("=" * 60)
print("DAY 19 - K-MEANS FINAL RESULTS")
print("=" * 60)

print(f"Total Customers : {len(df)}")
print(f"Features Used   : {X.shape[1]}")
print(f"Number of K     : {optimal_k}")
print(f"Final Inertia   : {kmeans.inertia_:.2f}")

print("\nCluster Sizes:")
print(cluster_counts)

print("\nCluster Centroids:")
display(centroids.round(2))

## 🔑 Key Findings

- K-Means groups observations without requiring a target label.
- Feature scaling is useful because K-Means uses distances.
- The Elbow Method helps investigate a suitable K.
- Centroids represent the center of each cluster.
- Income and spending behavior can be used to understand customer segments.
- PCA can provide a compact visualization of the clusters.

## 🧠 Key Learnings

- Unsupervised learning
- Clustering
- K-Means algorithm
- Centroids
- Inertia / WCSS
- Elbow Method
- Feature scaling
- Customer segmentation
- PCA visualization

## 🏁 Conclusion

K-Means is a useful unsupervised learning algorithm for discovering groups in unlabeled data. In this project, customers were segmented using annual income and spending score, with the Elbow Method used to investigate the number of clusters. The resulting segments were analyzed through cluster sizes, centroids, and visualizations.

## 📚 References

1. Scikit-learn — KMeans  
https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html

2. Scikit-learn — StandardScaler  
https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html

3. Scikit-learn — PCA  
https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

4. Scikit-learn — Clustering  
https://scikit-learn.org/stable/modules/clustering.html

5. Kaggle — Mall Customers Dataset  
https://www.kaggle.com/datasets/amisha0528/mall-customers-dataset

## 📅 30 Days of Machine Learning

### Day 19/30 — K-Means Clustering 🟢

Learning → Coding → Experimenting → Clustering → Analyzing 🚀